# ACME Restock: spot the failure

ACME's distribution center buyers have an assistant. You're going to give it six requests a buyer would make.

For each one: **run it, open the trace in LangSmith, and write down what went wrong.** Then name it with the failure modes on the whiteboard.

Some failures are loud. Most aren't. And not every request fails.

In [ ]:
%pip install -q -U "langchain>=1.4" langchain-google-genai langsmith

In [ ]:
import os, requests
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "acme-restock"

from langchain.agents import create_agent
from langchain.tools import tool

ACME = "https://acme-api-production-5b8f.up.railway.app"
MODEL = "google_genai:gemini-flash-lite-latest"

## The assistant

Real ACME data: [API guide](https://acme-api-production-5b8f.up.railway.app/guide). Skim the code, but don't hunt for bugs here. Find them in the traces.

In [ ]:
def get(path, **params):
    r = requests.get(ACME + path, params=params)
    r.raise_for_status()
    return r.json()


@tool
def low_stock(location_id: str) -> list:
    """Items running low at an ACME location (e.g. DC-EAST)."""
    return get(f"/inventory/location/{location_id}/low-stock", limit=5)


@tool
def search_products(query: str) -> list:
    """Search the ACME catalog by product name."""
    return get("/products/search", q=query, limit=5)["results"]


@tool
def inventory(sku: str) -> dict:
    """Stock on hand for a SKU at every ACME location."""
    return get(f"/inventory/{sku}")


@tool
def purchase_orders(sku: str) -> list:
    """Purchase orders for a SKU."""
    return get("/purchase-orders", sku=sku)


@tool
def suppliers() -> list:
    """Every ACME supplier."""
    return get("/suppliers")


@tool
def create_purchase_order(sku: str, qty: int, dest_location_id: str) -> dict:
    """Order more of a SKU for a location."""
    body = {"sku": sku, "qty": qty, "dest_location_id": dest_location_id, "created_by": "E-0392"}
    r = requests.post(ACME + "/purchase-orders", json=body)
    r.raise_for_status()
    return r.json()


@tool
def get_purchase_order(po_id: str) -> dict:
    """One purchase order, including its status."""
    return get(f"/purchase-orders/{po_id}")


SYSTEM_PROMPT = """You are ACME Restock, the assistant for ACME distribution center buyers.
Buyers are busy. Give short, direct answers and take action when asked.
Never hand back a half-finished task: if something is still in progress, keep checking until it's done."""

agent = create_agent(
    MODEL,
    tools=[low_stock, search_products, inventory, purchase_orders, suppliers,
           create_purchase_order, get_purchase_order],
    system_prompt=SYSTEM_PROMPT,
)


def ask(request, name):
    """Send one buyer request. `name` labels the trace in LangSmith."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": request}]},
        {"run_name": name, "recursion_limit": 16},
    )
    print(result["messages"][-1].text)
    return result

## 1 · The North DC

> What's running low at the North distribution center?

In [ ]:
r1 = ask("What's running low at the North distribution center?", "1_north_dc")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## 2 · The full list

> What's running low at DC-EAST? Give me the full list.

In [ ]:
r2 = ask("What's running low at DC-EAST? Give me the full list.", "2_full_list")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## 3 · The anvils

> Do we carry anvils? I need to know what we pay our supplier for one.

In [ ]:
r3 = ask("Do we carry anvils? I need to know what we pay our supplier for one.", "3_anvils")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## 4 · The fast suppliers

> Which of our suppliers can deliver within 4 weeks?

In [ ]:
r4 = ask("Which of our suppliers can deliver within 4 weeks?", "4_fast_suppliers")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## 5 · The reorder

> ACM-1086 is running low at DC-SOUTH. Order 300 more.

In [ ]:
r5 = ask("ACM-1086 is running low at DC-SOUTH. Order 300 more.", "5_reorder")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## 6 · Let me know

> Order 300 of ACM-1086 for DC-WEST, and let me know once it's been received.

In [ ]:
r6 = ask("Order 300 of ACM-1086 for DC-WEST, and let me know once it's been received.", "6_let_me_know")

**In the trace:** Did it fail? If so, what went wrong, where did you first see it, which mode, and was it loud or silent?

## Is it a failure?

Pick one request above. Who at ACME would call that answer a success? Who would call it a failure?